# Project 1

## Overview

In this project, you will build models that classify news articles by topic. You will represent each article with TF-IDF features (see 4) and train two classifiers introduced in this module: logistic regression and a multilayer perceptron (MLP). Transformers and pretrained language models are outside the scope of this project.

You will investigate whether models trained on one news dataset generalize to unseen in-domain articles, transfer to articles collected from a different source, and adapt when category definitions change. You will also study how regularization and limited target-domain training data affect performance.

Key concepts of this project include:
- construct a reproducible text-classification pipeline without evaluation-data leakage;
- compare a linear classifier with a nonlinear neural-network classifier;
- select hyperparameters using validation data and appropriate evaluation metrics;
- analyze in-domain performance, cross-dataset transfer, and category mismatch; and
- support quantitative results with error analysis and clear explanations.


### TODOs

[ ] Remove unnecessary (debug purpose) prints and displays

[x] Compute the training/validation accuracy for both LogRes & MLP

[ ] Compute macro-F1 for both LogRes & MLP

[ ] Compute F1 for both LogRes & MLP

[ ] Compute per-class precision for both LogRes & MLP

[ ] Compute Recall for both LogRes & MLP

[x] Refit the TF-IDF vectorizer & re-evaluate the model

[ ] Refactor training/validation loops

### Dependencies & Notebook Configurations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# for reproducibility
np.random.seed(0)
torch.manual_seed(0)

In [ ]:
# Use GPU if available
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

USE_CUDA = DEVICE.type  == "cuda"

print(f"DEVICE: {DEVICE}")
print(f"USE_CUDA: {USE_CUDA}")

## Data Load & Preprocessing

In [ ]:
# Set A
SOURCE_A_TRAIN = pd.read_csv("./dataset/dataset_a_train.csv")
SOURCE_A_TEST = pd.read_csv("./dataset/dataset_a_test.csv")

display(SOURCE_A_TRAIN.head())
display(SOURCE_A_TEST.head())

# Set B
SOURCE_B_TEST = pd.read_csv("./dataset/dataset_b_test.csv")

display(SOURCE_B_TEST.head())

# Set C
SOURCE_C_TRAIN = pd.read_csv("./dataset/dataset_c_train.csv")
SOURCE_C_TEST = pd.read_csv("./dataset/dataset_c_test.csv")

display(SOURCE_C_TRAIN.head())
display(SOURCE_C_TEST.head())

In [ ]:
LABELS = pd.DataFrame({
    "Label": range(1, 6),
    "Set A": ["Politics", "Sports", "Business", "Technology", "Entertainment"],
    "Set B": ["Politics", "Sports", "Business", "Technology", "Entertainment"],
    "Set C": ["World", "Sports", "Business", "Technology", None]
}).set_index("Label")

display(LABELS)

In [ ]:
LABELS_A_NUNIQUE = LABELS["Set A"].nunique()
LABELS_B_NUNIQUE = LABELS["Set B"].nunique()
LABELS_C_NUNIQUE = LABELS["Set C"].nunique()

_Note_: Labels 2, 3, and 4 have matching meanings across all three datasets. Label 1 does not: Politics in Data A and Data B is narrower than World in Data C. A political story may reasonably belong to the World category, but World also includes international events whose main topic is not politics. Data C also has no label 5.

## Part 2: Build an In-Domain Baseline on Data A

### Build a TF-IDF Table

In [ ]:
# Set A
a_train, a_validation = train_test_split(
    SOURCE_A_TRAIN,
    test_size=0.2,
    random_state=0,
    stratify=SOURCE_A_TRAIN["Label"]  # label-stratified split
)

print(SOURCE_A_TRAIN.shape)
display(a_train)
display(a_validation)

In [ ]:
BATCH_SIZE = 32

In [ ]:
vectorizer = TfidfVectorizer()  # default ngram_range is (1, 1), i.e. unigram

def concat_article(row):
    return row["Title"] + ' ' + row["Description"]

a_train_tfidf = vectorizer.fit_transform(a_train.apply(concat_article, axis=1))
a_validation_tfidf = vectorizer.transform(a_validation.apply(concat_article, axis=1))

a_train_tensor = torch.tensor(a_train_tfidf.toarray(), dtype=torch.float32).to(DEVICE)
a_validation_tensor = torch.tensor(a_validation_tfidf.toarray(), dtype=torch.float32).to(DEVICE)
print(a_train_tensor)
print(a_validation_tensor)

# CrossEntropyLoss expects zero-indexed labels, so label - 1
a_train_labels_tensor = torch.tensor((a_train["Label"] - 1).to_list(), dtype=torch.long).to(DEVICE)
a_validation_labels_tensor = torch.tensor((a_validation["Label"] - 1).to_list(), dtype=torch.long).to(DEVICE)
print(a_train_labels_tensor)
print(a_validation_labels_tensor)

In [ ]:
a_train_dataset = TensorDataset(
    a_train_tensor.cpu(),
    a_train_labels_tensor.cpu()
)
a_validation_dataset = TensorDataset(
    a_validation_tensor.cpu(),
    a_validation_labels_tensor.cpu()
)

a_train_loader = DataLoader(
    a_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,  # ensure the training data is shuffled in every epoch
    pin_memory=USE_CUDA
)
a_validation_loader = DataLoader(
    a_validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=USE_CUDA
)

### Logistic Regressor

In [ ]:
# Constants & Hyperparameters
MAX_EPOCH = 30
LEARNING_RATE = 0.01

In [ ]:
model = torch.nn.Linear(a_train_tensor.shape[1], LABELS_A_NUNIQUE).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)

In [ ]:
# TODO: Convert this into a function so we don't need to write down the whole bloody lines of code every single time we update it
training_losses = []
validation_losses = []

for epoch in range(MAX_EPOCH):
    print(f"Epoch: {epoch + 1:02}/{MAX_EPOCH}", end='')
    # Training
    model.train()

    training_loss = 0  # compute the cumulative loss within an epoch

    for i, (x, y) in enumerate(a_train_loader):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        logits = model(x)

        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # multiply by batch size to get the total loss for this batch
        training_loss += loss.item() * x.size(0)

    training_loss /= len(a_train_dataset)  # len(dataset) == len(tensor)
    training_losses.append(training_loss)

    # Validation
    model.eval()

    validation_loss = 0

    with torch.inference_mode():
        for i, (x, y) in enumerate(a_validation_loader):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            logits = model(x)

            loss = criterion(logits, y)

            validation_loss += loss.item() * x.size(0)

    validation_loss /= len(a_validation_dataset)

    validation_losses.append(validation_loss)

    print("\tDone!")

In [ ]:
plt.figure(figsize=(10, 7))

EPOCHS = range(1, MAX_EPOCH + 1)

plt.plot(EPOCHS, training_losses, label="Training Loss")
plt.plot(EPOCHS, validation_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("Training & Validation Losses (Logistic Regressor)")

plt.figtext(1, 0, f"Learning Rate: {LEARNING_RATE} / Batch Size: {BATCH_SIZE}", ha="right")

plt.legend()

plt.tight_layout()
plt.show()

### Custom Multi-Layer Perceptron

The model consists of the following layers:
1. Input Layer: input_dim
2. Hidden Layer 1: 256 neurons $\to$ ReLU activation $\to$ Dropout($p=0.1$)
3. Hidden Layer 2: 128 neurons $\to$ ReLU activation $\to$ Dropout($p=0.4$)
5. Output Layer: output_dim (number of labels)

Loss Function: Cross-Entropy

Optimizer: SGD

In [ ]:
class CustomMLP(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(256 ,128),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
# Constants & Hyperparameters
MAX_EPOCH = 35  # TODO: maybe we can do some early stopping during the training process

LEARNING_RATE = 0.021
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0017

In [ ]:
model = CustomMLP(a_train_tensor.shape[1], LABELS_A_NUNIQUE).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

In [ ]:
training_losses = []
validation_losses = []

training_accuracies = []
validation_accuracies = []

training_f1macros = []
training_f1s = []
training_precisions = []
training_recalls = []

validation_f1macros = []
validation_f1s = []
validation_precisions = []
validation_recalls = []

for epoch in range(MAX_EPOCH):
    print(f"Epoch: {epoch + 1:02}/{MAX_EPOCH}", end='')
    # Training
    model.train()

    training_loss = 0  # compute the cumulative loss within an epoch
    training_accuracy = 0

    training_predictions = []
    training_trues = []

    for i, (x, y) in enumerate(a_train_loader):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        logits = model(x)

        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # multiply by batch size to get the total loss for this batch
        training_loss += loss.item() * x.size(0)

        predictions = torch.argmax(logits, dim=1)

        training_predictions.extend(predictions.detach().cpu().numpy())
        training_trues.extend(y.cpu().numpy())

        training_accuracy += (predictions == y).sum().item()

    training_loss /= len(a_train_dataset)
    training_accuracy /= len(a_train_dataset)

    training_losses.append(training_loss)
    training_accuracies.append(training_accuracy)

    training_f1macro = f1_score(training_trues, training_predictions, average="macro", zero_division=0)
    training_f1macros.append(training_f1macro)

    training_f1 = f1_score(training_trues, training_predictions, average="weighted", zero_division=0)
    training_f1s.append(training_f1)

    training_precision = precision_score(training_trues, training_predictions, average="weighted", zero_division=0)
    training_precisions.append(training_precision)

    training_recall = recall_score(training_trues, training_predictions, average="weighted", zero_division=0)
    training_recalls.append(training_recall)

    # Validation
    model.eval()

    validation_loss = 0
    validation_accuracy = 0

    validation_predictions = []
    validation_trues = []

    with torch.inference_mode():
        for i, (x, y) in enumerate(a_validation_loader):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            logits = model(x)

            loss = criterion(logits, y)

            validation_loss += loss.item() * x.size(0)

            predictions = torch.argmax(logits, dim=1)

            validation_predictions.extend(predictions.detach().cpu().numpy())
            validation_trues.extend(y.cpu().numpy())

            validation_accuracy += (predictions == y).sum().item()

    validation_loss /= len(a_validation_dataset)
    validation_accuracy /= len(a_validation_dataset)

    validation_losses.append(validation_loss)
    validation_accuracies.append(validation_accuracy)

    validation_f1macro = f1_score(validation_trues, validation_predictions, average="macro", zero_division=0)
    validation_f1macros.append(validation_f1macro)

    validation_f1 = f1_score(validation_trues, validation_predictions, average="weighted", zero_division=0)
    validation_f1s.append(validation_f1)

    validation_precision = precision_score(validation_trues, validation_predictions, average="weighted", zero_division=0)
    validation_precisions.append(validation_precision)

    validation_recall = recall_score(validation_trues, validation_predictions, average="weighted", zero_division=0)
    validation_recalls.append(validation_recall)

    print("\tDone!")

In [ ]:
EPOCHS = range(1, MAX_EPOCH + 1)

In [ ]:
plt.figure(figsize=(10, 7))

plt.plot(EPOCHS, training_losses, label="Training Loss")
plt.plot(EPOCHS, validation_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("Training & Validation Losses (Custom MLP)")

plt.figtext(1, 0, f"Learning Rate: {LEARNING_RATE} / Batch Size: {BATCH_SIZE} / Momentum: {MOMENTUM} / Weight Decay: {WEIGHT_DECAY}", ha="right")

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))

plt.plot(EPOCHS, training_accuracies, label="Training Accuracy")
plt.plot(EPOCHS, validation_accuracies, label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.title("Training & Validation Accuracies (Custom MLP)")

plt.figtext(1, 0, f"Learning Rate: {LEARNING_RATE} / Batch Size: {BATCH_SIZE} / Momentum: {MOMENTUM} / Weight Deacy: {WEIGHT_DECAY}", ha="right")

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))

plt.plot(
    EPOCHS,
    training_f1macros,
    label="Training Macro-F1"
)

plt.plot(
    EPOCHS,
    validation_f1macros,
    label="Validation Macro-F1"
)

plt.plot(
    EPOCHS,
    training_f1s,
    label="Training F1 (Weighted)"
)

plt.plot(
    EPOCHS,
    validation_f1s,
    label="Validation F1 (Weighted)"
)

plt.xlabel("Epoch")
plt.ylabel("F1 Score")

plt.title("Training & Validation Macro-F1/F1 Scores")

plt.figtext(1, 0, f"Learning Rate: {LEARNING_RATE} / Batch Size: {BATCH_SIZE} / Momentum: {MOMENTUM} / Weight Deacy: {WEIGHT_DECAY}", ha="right")

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))

plt.plot(
    EPOCHS,
    training_precisions,
    label="Training Precision"
)

plt.plot(
    EPOCHS,
    validation_precisions,
    label="Validation Precision"
)

plt.plot(
    EPOCHS,
    training_recalls,
    label="Training Recall"
)

plt.plot(
    EPOCHS,
    validation_recalls,
    label="Validation Recall"
)

plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Training & Validation Precision/Recall")

plt.ylim(0, 1)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
cm = confusion_matrix(
    validation_trues,
    validation_predictions
)

cm_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS["Set A"].to_list())  # overriding display_labels to show the actual labels
cm_display.plot(values_format='d')

plt.title("Validation Confusion Matrix (Custom MLP)")

plt.figtext(1, 0, f"Learning Rate: {LEARNING_RATE} / Batch Size: {BATCH_SIZE} / Momentum: {MOMENTUM} / Weight Deacy: {WEIGHT_DECAY}", ha="right")

plt.tight_layout()
plt.show()

## Refit & Test

In [ ]:
a_test_tfidf = vectorizer.transform(SOURCE_A_TEST.apply(concat_article, axis=1))

a_test_tensor = torch.tensor(a_test_tfidf.toarray(), dtype=torch.float32).to(DEVICE)
print(a_test_tensor)

a_test_labels_tensor = torch.tensor((SOURCE_A_TEST["Label"] - 1).to_list(), dtype=torch.long).to(DEVICE)
print(a_test_labels_tensor)

In [ ]:
a_test_dataset = TensorDataset(
    a_test_tensor.cpu(),
    a_test_labels_tensor.cpu()
)

a_test_loader = DataLoader(
    a_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=USE_CUDA
)

In [ ]:
testing_predictions = []
testing_trues = []

model.eval()

with torch.inference_mode():
    for i, (x, y) in enumerate(a_test_loader):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        logits = model(x)

        p = torch.argmax(logits, dim=1)

        testing_predictions.extend(p.detach().cpu().numpy())
        testing_trues.extend(y.cpu().numpy())

testing_accuracy = accuracy_score(testing_trues, testing_predictions)

print(testing_accuracy)
